# D08_S03 – Refugee Camp AC microgrid in pandapower, mapped at Bidibidi

**Task.** Recreate the D7 AC refugee-camp design in pandapower, place the buses at Bidibidi Refugee Camp
(3.523658° N, 31.338941° E), and plot the bus voltages and line loadings on a map.

**What you'll end up with**
1. A pandapower model of the 24-home, 120 V single-phase AC network.
2. Tables of the bus voltages, line currents, loadings and losses, checked against the ±10 % rule.
3. A check that pandapower agrees with your own Gauss-Seidel code from D7.
4. An interactive map (`bidibidi_powerflow_map.html`) and a static figure for slides.

---
### Step 0 – Setup (do this once)

**Google Colab (easiest):** open a new notebook, upload this file, and run the install cell below.

**On your own laptop:** in a terminal run
```
pip install pandapower folium matplotlib
jupyter notebook
```
then open this notebook. Run the cells **in order** from top to bottom (Shift + Enter).

In [ ]:
# Only needed the first time (or every time in Colab). Remove the '#' to run it.
# !pip install pandapower folium matplotlib

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pandapower as pp

print("pandapower version:", pp.__version__)

### Step 1 – The design we are recreating (from D4 and D7)

* The camp is a **5 × 5 grid of 50 m × 50 m squares**. The **centre square** holds the PV, battery and
  single-phase inverter (the *hub*). The other 24 squares are homes.
* As in the DC design (D04_S02), the hub feeds **4 identical radial feeders of 6 homes each**. Every
  connection is between neighbouring squares, so each line span is **50 m** (Manhattan distance).
* Line data (D07_S01): **R = 0.37 Ω per span**, **X/R = 1**, so X = 0.37 Ω.
* Loads: **ADMD 300 W per home**, constant power, **pf = 0.95** lagging, so Q = P·tan(cos⁻¹0.95) ≈ 98.6 var.
* Supply: **120 V nominal**, inverter set to **130 V** (the same `VSlack` as your GS script).
* Regulator: every voltage must be within **±10 % of 120 V**, i.e. 108 V to 132 V (0.90 to 1.10 pu).

These are the same numbers as your D7 Gauss-Seidel script. Changing any of them below changes the whole notebook.

In [ ]:
# ---- Inputs (identical to the D7 Gauss-Seidel script) ----
V_NOM   = 120.0                       # nominal voltage [V]
V_SLACK = 130.0                       # inverter (slack bus) set-point [V]
P_HOME  = 300.0                       # ADMD per home [W]
PF      = 0.95                        # power factor (lagging)
Q_HOME  = P_HOME * np.tan(np.arccos(PF))   # reactive power per home [var]
R_SPAN  = 0.37                        # resistance of ONE 50 m span [ohm]
X_OVER_R = 1.0
X_SPAN  = R_SPAN * X_OVER_R           # reactance of one span [ohm]
SPAN_M  = 50.0                        # span length [m]
V_TOL   = 0.10                        # regulator limit: +/-10 % of nominal

# ---- Cable thermal rating (ASSUMPTION - replace with your cable's datasheet value) ----
# Back out the copper cross-section from R = rho*L/A, then use the ampacity curve fit
# I = 9.2718 * A^0.7075 from D07_S02 (that fit is for ACSR overhead conductor, so treat it as an estimate).
RHO_CU  = 1.72e-8                                  # copper resistivity [ohm m]
A_MM2   = RHO_CU * SPAN_M / R_SPAN * 1e6           # cross-section [mm^2]
I_RATED = 9.2718 * A_MM2**0.7075                   # thermal rating [A]

print(f"Q per home        = {Q_HOME:.1f} var")
print(f"Conductor area    = {A_MM2:.2f} mm^2 (nearest standard size 2.5 mm^2)")
print(f"Assumed rating    = {I_RATED:.1f} A")

### Step 2 – How a single-phase network fits into pandapower

pandapower always models a **balanced three-phase** network: `vn_kv` is a line-to-line voltage and `p_mw`
is a three-phase total. Our camp is single-phase. The trick is that the per-unit equations are identical
whichever way you read them, so:

| What we enter | Value | Why |
|---|---|---|
| `vn_kv` | 0.120 | Our 120 V becomes the voltage base. Units are **kV**, not V. |
| `p_mw`, `q_mvar` per load | 300 × 10⁻⁶, 98.6 × 10⁻⁶ | The full single-phase load, in **MW / Mvar**. |
| `r_ohm_per_km`, `x_ohm_per_km` | 0.37 / 0.05 = 7.4 Ω/km | The whole span resistance, as in the GS code. |

With these inputs, **`vm_pu`, the angles and the losses come out exactly as in the single-phase GS
solution**. The one thing that changes is the current: pandapower reports `i_ka = S / (√3·V)`, but the real
single-phase current is `S / V`. So:

* real current = `i_ka × 1000 × √3` A
* set `max_i_ka = I_RATED / √3 / 1000` so that `loading_percent` uses the real current.

(Same idea as the D08_S01 hint for the three-bus problem: **pandapower wants real ohms, not pu.** Here we
already have real ohms, so `length_km = 0.05` and R per km = 0.37 / 0.05.)

### Step 3 – Lay out the feeders on the 5 × 5 grid

Grid cells are `(row, col)`, with row 0 on the **north** edge and col 0 on the **west** edge. The hub is
`(2, 2)`. We write down feeder 1 as a snake through the north-west block, then rotate it by 90° three
times to get the other three feeders, which keeps the 4 feeders identical.

In [ ]:
HUB_CELL = (2, 2)

# Feeder 1: hub -> (1,2) -> (1,1) -> (1,0) -> (0,0) -> (0,1) -> (0,2)
feeder_1 = [(1, 2), (1, 1), (1, 0), (0, 0), (0, 1), (0, 2)]

def rotate90(cell):
    # Rotate a cell 90 degrees clockwise about the centre of the 5x5 grid
    r, c = cell
    return (c, 4 - r)

feeders = [feeder_1]
for _ in range(3):
    feeders.append([rotate90(cell) for cell in feeders[-1]])

# --- sanity checks: 24 different homes, hub not used, every span joins neighbouring squares (50 m)
homes = [cell for f in feeders for cell in f]
assert len(set(homes)) == 24 and HUB_CELL not in homes
for f in feeders:
    path = [HUB_CELL] + f
    for a, b in zip(path[:-1], path[1:]):
        assert abs(a[0] - b[0]) + abs(a[1] - b[1]) == 1, (a, b)

# --- print the layout: F<feeder>.<home number along the feeder>
grid = [["  HUB "] * 5 for _ in range(5)]
for fi, f in enumerate(feeders, start=1):
    for hi, (r, c) in enumerate(f, start=1):
        grid[r][c] = f" F{fi}.{hi} "
print("North at the top")
for row in grid:
    print("|".join(row))

### Step 4 – Turn grid squares into latitude / longitude

We put the **hub at the Bidibidi coordinate** and each home at the centre of its square. Over a few hundred
metres the Earth is flat enough to use:

* 1° of latitude ≈ 111 195 m (= 2π × 6371 km / 360)
* 1° of longitude ≈ 111 195 m × cos(latitude)

⚠️ **Coordinate order trap.** Google Maps gives **(lat, lon)** = (3.523658, 31.338941). pandapower's
`geodata` is **(x, y) = (lon, lat)**. folium wants **[lat, lon]**. If you swap them, your camp appears in the
Sahara instead of Uganda.

In [ ]:
LAT0, LON0 = 3.523658, 31.338941          # Bidibidi Refugee Camp (lat, lon) - the hub
M_PER_DEG  = 2 * np.pi * 6_371_000 / 360  # metres per degree of latitude

def cell_to_latlon(cell):
    r, c = cell
    north_m = (HUB_CELL[0] - r) * SPAN_M      # row 0 is north
    east_m  = (c - HUB_CELL[1]) * SPAN_M      # col 0 is west
    lat = LAT0 + north_m / M_PER_DEG
    lon = LON0 + east_m / (M_PER_DEG * np.cos(np.radians(LAT0)))
    return float(lat), float(lon)

print("Hub        :", cell_to_latlon(HUB_CELL))
print("NW corner  :", cell_to_latlon((0, 0)))

### Step 5 – Build the pandapower network

The order is the same as in D08_S01: empty network → buses → slack (`ext_grid`) → loads → lines.
We also keep a small table `meta` of each bus's feeder, home number and coordinates. It makes the plotting
independent of how your pandapower version stores geodata.

In [ ]:
net = pp.create_empty_network(name="Bidibidi 120 V AC microgrid", f_hz=50.0)
vn_kv = V_NOM / 1000                         # 0.12 kV

meta = []                                    # one row per bus, for tables and plots

# --- hub bus + inverter as the slack bus (fixes |V| = 130 V, angle = 0)
lat, lon = cell_to_latlon(HUB_CELL)
hub = pp.create_bus(net, vn_kv=vn_kv, name="Hub", geodata=(lon, lat))   # (x, y) = (lon, lat)
pp.create_ext_grid(net, bus=hub, vm_pu=V_SLACK / V_NOM, va_degree=0.0, name="Inverter")
meta.append(dict(bus=hub, name="Hub", feeder=0, home=0, lat=lat, lon=lon))

# --- the four feeders
for fi, feeder in enumerate(feeders, start=1):
    prev_bus, prev_lat, prev_lon = hub, lat, lon
    for hi, cell in enumerate(feeder, start=1):
        h_lat, h_lon = cell_to_latlon(cell)
        name = f"F{fi}-H{hi}"

        b = pp.create_bus(net, vn_kv=vn_kv, name=name, geodata=(h_lon, h_lat))

        # constant-power load (pandapower's default load model)
        pp.create_load(net, bus=b, p_mw=P_HOME / 1e6, q_mvar=Q_HOME / 1e6, name=f"Home {name}")

        # 50 m span: real ohms per km, no capacitance (short LV line)
        pp.create_line_from_parameters(
            net, from_bus=prev_bus, to_bus=b,
            length_km=SPAN_M / 1000,
            r_ohm_per_km=R_SPAN / (SPAN_M / 1000),
            x_ohm_per_km=X_SPAN / (SPAN_M / 1000),
            c_nf_per_km=0.0,
            max_i_ka=I_RATED / np.sqrt(3) / 1000,      # see Step 2 for the sqrt(3)
            name=f"F{fi} line {hi-1}-{hi}",
            geodata=[(prev_lon, prev_lat), (h_lon, h_lat)])

        meta.append(dict(bus=b, name=name, feeder=fi, home=hi, lat=h_lat, lon=h_lon))
        prev_bus, prev_lat, prev_lon = b, h_lat, h_lon

meta = pd.DataFrame(meta).set_index("bus")
print(net)          # should show 25 buses, 24 loads, 24 lines, 1 ext_grid

### Step 6 – Run the AC power flow

`pp.runpp` uses Newton-Raphson by default (your D7 code used Gauss-Seidel; both should give the same
answer). `numba=False` just stops a warning if numba isn't installed.

In [ ]:
pp.runpp(net, numba=False)
print("Converged:", net.converged)

### Step 7 – Results in real units, and the regulator check

In [ ]:
# ---- Bus results
bus_res = meta.join(net.res_bus[["vm_pu", "va_degree"]])
bus_res["V (V)"] = bus_res["vm_pu"] * V_NOM
bus_res["OK (±10%)"] = bus_res["vm_pu"].between(1 - V_TOL, 1 + V_TOL)

# ---- Line results
line_res = net.line[["name", "from_bus", "to_bus"]].join(
    net.res_line[["i_ka", "loading_percent", "p_from_mw", "pl_mw", "ql_mvar"]])
line_res["I (A)"]     = line_res["i_ka"] * 1000 * np.sqrt(3)   # real single-phase current
line_res["P_send (W)"] = line_res["p_from_mw"] * 1e6
line_res["loss (W)"]  = line_res["pl_mw"] * 1e6
line_res["loss (var)"] = line_res["ql_mvar"] * 1e6

pd.set_option("display.precision", 3)
print("Feeder 1 (all four feeders are identical):")
print(bus_res[bus_res.feeder.isin([0, 1])][["name", "vm_pu", "V (V)", "va_degree", "OK (±10%)"]].to_string())
print()
print(line_res[line_res["name"].str.startswith("F1")][["name", "I (A)", "loading_percent",
                                                          "P_send (W)", "loss (W)"]].to_string())

In [ ]:
# ---- System summary and power balance
P_supply = net.res_ext_grid.p_mw.sum() * 1e6
Q_supply = net.res_ext_grid.q_mvar.sum() * 1e6
P_load   = net.res_load.p_mw.sum() * 1e6
P_loss   = net.res_line.pl_mw.sum() * 1e6

print(f"Inverter output  : {P_supply:8.1f} W  + j{Q_supply:7.1f} var")
print(f"Total load       : {P_load:8.1f} W   (24 x {P_HOME:.0f} W)")
print(f"Line losses      : {P_loss:8.1f} W   ({100 * P_loss / P_supply:.1f} % of power sent)")
print(f"Balance check    : supply - load - losses = {P_supply - P_load - P_loss:.2e} W  (should be ~0)")
print()
print(f"Lowest voltage   : {bus_res['V (V)'].min():.1f} V  (limit {V_NOM * (1 - V_TOL):.0f} V)")
print(f"Highest voltage  : {bus_res['V (V)'].max():.1f} V  (limit {V_NOM * (1 + V_TOL):.0f} V)")
bad = bus_res[~bus_res["OK (±10%)"]]
print(f"Buses outside ±10 %: {len(bad)} of {len(bus_res)} -> {', '.join(bad['name'])}")
print(f"Max line loading : {line_res['loading_percent'].max():.0f} % (assumed rating {I_RATED:.1f} A)")

### Step 8 – Validate against your own Gauss-Seidel code (D7)

Always check a new tool against a method you trust (D08_S01 learning objective 3). Below is your D7 GS loop
for one feeder, written compactly in per unit (S_base = 1000 VA, V_base = 120 V). Feeder 1 in pandapower
should match it to about 6 decimal places.

In [ ]:
S_BASE = 1000.0
Z_BASE = V_NOM**2 / S_BASE
Y = 1 / ((R_SPAN + 1j * X_SPAN) / Z_BASE)        # line admittance [pu]
S_h = (P_HOME + 1j * Q_HOME) / S_BASE             # home load [pu]
V0 = V_SLACK / V_NOM

V = np.ones(7, dtype=complex); V[0] = V0          # V[0] = slack, V[1..6] = homes
for it in range(10_000):
    V_old = V.copy()
    for i in range(1, 7):
        neighbours = V[i - 1] + (V[i + 1] if i < 6 else 0)
        n_lines = 2 if i < 6 else 1                     # the last home has only one line
        V[i] = (-np.conj(S_h) / np.conj(V[i]) + Y * neighbours) / (n_lines * Y)
    if np.sum(np.abs(V - V_old)) < 1e-10:
        break

pp_vm = bus_res[bus_res.feeder.isin([0, 1])].sort_values("home")["vm_pu"].values
compare = pd.DataFrame({"GS |V| (pu)": np.abs(V), "pandapower vm_pu": pp_vm,
                        "difference": np.abs(V) - pp_vm}, index=[f"node {k}" for k in range(7)])
print(f"GS converged in {it + 1} iterations")
print(compare.to_string())

### Step 9 – Plot the power flow on a real map (interactive)

We use **folium**, which draws on OpenStreetMap or satellite tiles and needs no API key. Each line is
coloured by **loading (%)** (light to dark blue) and each bus by **voltage (pu)** (green = OK, red = outside ±10 %). Hover over a line or a dot to see its values.
Use the layer button (top right) to switch to satellite view and see the camp itself.

In [ ]:
import folium
from branca.colormap import LinearColormap

# Buses: green = fine, orange = at the limit, red = outside it.  Lines: light -> dark blue = more loaded
v_cmap = LinearColormap(["#b2182b", "#f4a582", "#1a9850", "#f4a582", "#b2182b"],
                        index=[0.80, 0.90, 1.00, 1.10, 1.20], vmin=0.80, vmax=1.20,
                        caption="Bus voltage (pu)  -  allowed 0.90 to 1.10")
l_cmap = LinearColormap(["#c6dbef", "#6baed6", "#08306b"], index=[0, 50, 100], vmin=0, vmax=100,
                        caption="Line loading (%)")

m = folium.Map(location=[LAT0, LON0], zoom_start=18, max_zoom=21, tiles="OpenStreetMap")
folium.TileLayer(
    tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",
    attr="Esri World Imagery", name="Satellite", max_zoom=21).add_to(m)

# --- lines first so the bus dots sit on top
for idx, row in line_res.iterrows():
    a, b = meta.loc[row.from_bus], meta.loc[row.to_bus]
    loading = row.loading_percent
    folium.PolyLine(
        locations=[[a.lat, a.lon], [b.lat, b.lon]],
        color=l_cmap(min(loading, 100)), weight=7, opacity=0.9,
        tooltip=f"{row['name']}: {row['I (A)']:.1f} A, {loading:.0f} % loaded, loss {row['loss (W)']:.1f} W",
    ).add_to(m)

# --- buses
for bus, row in bus_res.iterrows():
    is_hub = row.feeder == 0
    folium.CircleMarker(
        location=[row.lat, row.lon], radius=12 if is_hub else 8,
        color="black", weight=2 if is_hub else 1,
        fill=True, fill_color=v_cmap(float(np.clip(row.vm_pu, 0.80, 1.20))), fill_opacity=1.0,
        tooltip=f"{row['name']}: {row['V (V)']:.1f} V ({row.vm_pu:.3f} pu, {row.va_degree:.1f}°)"
                + ("" if row["OK (±10%)"] else "  OUTSIDE ±10 %"),
    ).add_to(m)

v_cmap.add_to(m)
l_cmap.add_to(m)
folium.LayerControl().add_to(m)

m.save("bidibidi_powerflow_map.html")   # open this file in any browser
m                                        # shows the map inside Jupyter / Colab

### Step 10 – Static figure for your slides / report

Same information as the map, drawn with matplotlib on longitude/latitude axes (no internet needed).

In [ ]:
from matplotlib.collections import LineCollection
from matplotlib.colors import LinearSegmentedColormap, Normalize
from matplotlib.ticker import MaxNLocator, FormatStrFormatter

# Voltage colours: green at 1.0 pu, orange at the 0.9/1.1 limits, red beyond (same as the folium map)
v_colours = LinearSegmentedColormap.from_list(
    "vlim", [(0.00, "#b2182b"), (0.25, "#f4a582"), (0.50, "#1a9850"), (0.75, "#f4a582"), (1.00, "#b2182b")])
v_norm = Normalize(0.80, 1.20)
# Loading colours: light blue (lightly loaded) to dark blue (100 %+)
l_colours = LinearSegmentedColormap.from_list("load", ["#c6dbef", "#6baed6", "#08306b"])
l_norm = Normalize(0, 100)

fig, ax = plt.subplots(figsize=(8, 8.5), layout="constrained")
segs = [[(meta.loc[r.from_bus].lon, meta.loc[r.from_bus].lat),
         (meta.loc[r.to_bus].lon,   meta.loc[r.to_bus].lat)] for _, r in line_res.iterrows()]
lc = LineCollection(segs, cmap=l_colours, norm=l_norm, linewidths=7, zorder=1)
lc.set_array(line_res["loading_percent"].clip(upper=100).values)
ax.add_collection(lc)

sc = ax.scatter(bus_res.lon, bus_res.lat, c=bus_res.vm_pu, cmap=v_colours, norm=v_norm,
                s=np.where(bus_res.feeder == 0, 450, 200), edgecolors="black", zorder=3)
for _, r in bus_res.iterrows():
    ax.annotate(f"{r['V (V)']:.0f} V", (r.lon, r.lat), textcoords="offset points",
                xytext=(0, 13), ha="center", fontsize=8, zorder=4)

fig.colorbar(sc, ax=ax, location="right", shrink=0.8, extend="both",
             label="Bus voltage (pu)  -  allowed 0.90 to 1.10")
fig.colorbar(lc, ax=ax, location="bottom", shrink=0.8, extend="max", label="Line loading (%)")
ax.set_aspect(1 / np.cos(np.radians(LAT0)))     # so 50 m looks the same in both directions
ax.margins(0.12)
ax.xaxis.set_major_locator(MaxNLocator(4)); ax.yaxis.set_major_locator(MaxNLocator(5))
ax.xaxis.set_major_formatter(FormatStrFormatter("%.4f")); ax.yaxis.set_major_formatter(FormatStrFormatter("%.4f"))
ax.set_xlabel("Longitude (°E)"); ax.set_ylabel("Latitude (°N)")
ax.set_title("Bidibidi 120 V AC microgrid: power flow\nADMD 300 W/home, pf 0.95, inverter at 130 V")
plt.savefig("bidibidi_powerflow_static.png", dpi=200)
plt.show()

### Step 11 (optional) – pandapower's built-in map plot

pandapower can also draw results itself with plotly (`pip install plotly`). Depending on your pandapower
version, `on_map=True` may need a free Mapbox token. If this cell fails, use the folium map from Step 9,
which shows the same information.

In [ ]:
try:
    from pandapower.plotting.plotly import pf_res_plotly
    pf_res_plotly(net, on_map=True, climits_volt=(0.9, 1.1), climits_load=(0, 100),
                  filename="pandapower_pf_res_plotly.html", auto_open=False)
except Exception as e:
    print("Built-in plotly map not available here:", e)

### Step 12 – Interpreting the result

* **Voltages.** The inverter holds the hub at 130 V (1.083 pu). Volts drop along each feeder:
  ≈121.5 → 114.3 → 108.5 → 104.1 → 101.2 → **99.7 V** at the last home. **Homes 4–6 on every feeder
  (12 of the 24) are below 108 V**, so this design does **not** meet the regulator's ±10 % rule at 300 W ADMD.
  Raising the set-point doesn't help: it is already near the 132 V upper limit.
* **Why it's worse than DC.** The drop per span is roughly I(R cos φ + X sin φ). With X = R and pf 0.95 that
  is about 1.26·I·R, so the same copper gives ~25 % more drop than in DC. On top of that, the DC design used a
  diversified ADMD of 200 W, and here every home is 300 W.
* **Loading.** The first span of each feeder carries ≈17.6 A, around 105 % of the *assumed* 16.8 A rating.
  Check this against your real cable datasheet.
* **Losses.** ≈1.2 kW of the ≈8.4 kW sent (≈14 %), mostly in the first two spans.
* **Next step (D7 challenge).** Change `V_NOM`/`V_SLACK` to 230 V and increase `P_HOME` until the lowest
  voltage just reaches 0.90 pu. That gives you the maximum ADMD on the original lines.

### Common mistakes
| Symptom | Likely cause |
|---|---|
| Voltages all ≈ 1.000 pu | `vn_kv=120` instead of `0.12` (you built a 120 kV network), or loads entered in W instead of MW |
| Power flow doesn't converge | Loads 10⁶× too large (W entered as MW) |
| Map shows the Sahara / empty sea | Latitude and longitude swapped |
| Currents √3 too small vs. GS | Forgot the `× √3` single-phase correction (Step 2) |
| `KeyError` on `geodata` | Very old pandapower: update with `pip install -U pandapower` |